# Task 3: LM Exploration and Analysis

**Model:** `distilgpt2`  
**Tools:** Hugging Face Transformers, LangChain, Pandas, Matplotlib

## 1. Setup

In [ ]:
import re
import time
import textwrap
from typing import Dict, List

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 180)

## 2. Load Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
    pad_token_id=tokenizer.eos_token_id,
)

In [ ]:
try:
    from langchain_huggingface import HuggingFacePipeline
except ImportError:
    from langchain_community.llms import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=generator)
type(llm).__name__

## 3. Test Inputs

In [ ]:
scenarios: List[Dict[str, object]] = [
    {
        "scenario": "Context understanding",
        "prompt": (
            "Context: Priya is building an NLP project using a small GPT-style "
            "language model. Question: What is Priya building? Answer:"
        ),
        "expected_keywords": ["NLP", "project", "language", "model"],
    },
    {
        "scenario": "Summarization",
        "prompt": (
            "Summarize in one sentence: Language models can generate fluent text, "
            "but they may produce incorrect facts, biased wording, or unsupported claims. Summary:"
        ),
        "expected_keywords": ["language", "models", "incorrect", "facts"],
    },
    {
        "scenario": "Creative generation",
        "prompt": "Write a short introduction for an AI-driven NLP project:",
        "expected_keywords": ["AI", "NLP", "project"],
    },
    {
        "scenario": "Reasoning",
        "prompt": (
            "Maya has 3 notebooks. She buys 4 more and gives 2 to her friend. "
            "How many notebooks does Maya have now? Answer:"
        ),
        "expected_keywords": ["5", "notebooks"],
    },
    {
        "scenario": "Responsible AI",
        "prompt": "Give neutral advice about using AI tools responsibly in education:",
        "expected_keywords": ["responsibly", "verify", "learning", "privacy"],
    },
]

prompts_df = pd.DataFrame(scenarios)
prompts_df[["scenario", "prompt", "expected_keywords"]]

## 4. Generate Responses

In [ ]:
def clean_generated_text(prompt: str, generated_text: str) -> str:
    if generated_text.startswith(prompt):
        return generated_text[len(prompt):].strip()
    return generated_text.strip()


def keyword_score(text: str, keywords: List[str]) -> float:
    text_lower = text.lower()
    matches = sum(1 for keyword in keywords if keyword.lower() in text_lower)
    return round(matches / len(keywords), 2) if keywords else 0.0


def lexical_diversity(text: str) -> float:
    tokens = re.findall(r"[A-Za-z']+", text.lower())
    return round(len(set(tokens)) / len(tokens), 2) if tokens else 0.0


def generate_response(prompt: str) -> Dict[str, object]:
    start = time.perf_counter()
    output = generator(prompt, num_return_sequences=1)[0]["generated_text"]
    latency = round(time.perf_counter() - start, 2)
    response = clean_generated_text(prompt, output)

    return {
        "response": response,
        "latency_seconds": latency,
        "word_count": len(response.split()),
        "lexical_diversity": lexical_diversity(response),
    }

In [ ]:
records = []

for item in scenarios:
    result = generate_response(item["prompt"])
    records.append(
        {
            "scenario": item["scenario"],
            "prompt": item["prompt"],
            "response": result["response"],
            "keyword_score": keyword_score(result["response"], item["expected_keywords"]),
            "word_count": result["word_count"],
            "lexical_diversity": result["lexical_diversity"],
            "latency_seconds": result["latency_seconds"],
        }
    )

results_df = pd.DataFrame(records)
results_df[["scenario", "response", "keyword_score", "word_count", "lexical_diversity", "latency_seconds"]]

## 5. Metrics

In [ ]:
metric_cols = ["keyword_score", "word_count", "lexical_diversity", "latency_seconds"]
results_df[["scenario", *metric_cols]]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

charts = [
    ("keyword_score", "Keyword Score", "#4C78A8"),
    ("word_count", "Response Length", "#59A14F"),
    ("lexical_diversity", "Lexical Diversity", "#F28E2B"),
    ("latency_seconds", "Latency", "#B07AA1"),
]

for ax, (column, title, color) in zip(axes.flat, charts):
    sns.barplot(data=results_df, x="scenario", y=column, ax=ax, color=color)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)

axes[0, 0].set_ylim(0, 1)
axes[1, 0].set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 6. Response Review

In [ ]:
for _, row in results_df.iterrows():
    print("=" * 90)
    print(f"Scenario: {row['scenario']}")
    print("\nPrompt:")
    print(textwrap.fill(row["prompt"], width=100))
    print("\nResponse:")
    print(textwrap.fill(row["response"], width=100))
    print(
        f"\nKeyword score: {row['keyword_score']} | "
        f"Words: {row['word_count']} | "
        f"Latency: {row['latency_seconds']}s"
    )

## 7. Prompt Comparison

In [ ]:
comparison_prompts = [
    {
        "prompt_type": "Basic",
        "prompt": "Explain why language models need evaluation:",
    },
    {
        "prompt_type": "Structured",
        "prompt": (
            "For a beginner NLP student, explain in 3 short bullet points why "
            "language models need evaluation. Mention factuality, bias, and usefulness. Answer:"
        ),
    },
]

comparison_records = []

for item in comparison_prompts:
    result = generate_response(item["prompt"])
    comparison_records.append({**item, **result})

comparison_df = pd.DataFrame(comparison_records)
comparison_df[["prompt_type", "prompt", "response", "word_count", "lexical_diversity"]]

## 8. Findings

In [ ]:
findings = pd.DataFrame(
    [
        {
            "area": "Strength",
            "observation": "The model can create fluent text for open-ended and creative prompts.",
        },
        {
            "area": "Context",
            "observation": "It uses some prompt context, but exact question answering is not always reliable.",
        },
        {
            "area": "Limitation",
            "observation": "Reasoning and factual accuracy need human checking.",
        },
        {
            "area": "Ethics",
            "observation": "Outputs should be reviewed for bias, hallucination, privacy, and misuse risk.",
        },
        {
            "area": "Improvement",
            "observation": "A stronger instruction-tuned model or retrieval-based system would improve results.",
        },
    ]
)

findings

## Conclusion

The selected LM is useful for demonstrating text generation, prompt sensitivity, and basic NLP experimentation. It performs best on fluent open-ended generation and needs extra checks for reasoning, factuality, and responsible use.